# 0. From a Measurement Set to an image

This tutorial takes a MeerKAT L-band Measurement Set (MS) to a calibrated,
deconvolved image with two commands, `rang-core run config.yaml my.ms` and
`gemclean`. It covers each section of the configuration, everything the run
writes and what rang-core refuses. It does not explain the physics: notebooks
1 to 4 and {doc}`../../concepts/index` do that.

**The data** are A1 small, rang's simulated MeerKAT observation
({gh}`research/a1-simtruth.md`). It has 16 antennas, 64 channels of 13.4 MHz
from 856 to 1712 MHz, four linear correlations and 104 integrations of 8 s in
23 scans. The fields are the flux and bandpass calibrator J1939−6342 (PKS
1934−638), a polarisation calibrator, a gain calibrator and a target of 40
points and 15 Gaussians. Delays, bandpasses, gains, leakages, RFI (with its
mask) and thermal noise are simulated with known values, so every result
below is compared with the truth. The MS is written from seeds through rang's
own table writer, 41 MB in about 4 s.

**Cost:** the whole notebook takes well under a minute on a laptop with 2
threads. The commands are the ones you would type in a shell. In this
notebook `sh(...)` runs them, prints each command line, and runs `rang-core`
and `gemclean` from this kernel's Python.

**What you need:** a source checkout with rang-core installed
(`pip install -e '.[notebook,imaging]'`). The A1 generator,
`examples/a1_simtruth.py`, and its antenna table `data/meerkat.itrf.txt` are
in the checkout, not in the wheel. On your own MS you need only
`pip install rang-core`.

In [ ]:
%matplotlib inline
import json
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("RAYON_NUM_THREADS", "2")  # the core's threads; set before the first core call

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS

from rang import core_ms
from rang.formats import flagversions
from rang.notebook import data

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "a1_simtruth.py").is_file())
# Everything is written here: <RANG_NOTEBOOK_CACHE>/ms-to-image, emptied at the start of each run.
WORK = data.default_cache_dir() / "ms-to-image"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
COMMANDS = {
    "rang-core": [sys.executable, "-m", "rang.core_cli"],
    "gemclean": [sys.executable, "-m", "rang.gemclean"],
    "python": [sys.executable],
}


def sh(line, expect=0, tail=None):
    # Run one command line in WORK as a shell would, print it and its output.
    print("$", line)
    argv = shlex.split(line)
    argv = COMMANDS.get(argv[0], [argv[0]]) + argv[1:]
    p = subprocess.run(argv, cwd=WORK, capture_output=True, text=True, encoding="utf-8")
    lines = (p.stdout + p.stderr).rstrip().splitlines()
    print("\n".join(lines[-tail:] if tail else lines))
    print(f"[exit {p.returncode}]")
    assert p.returncode == expect, f"expected exit {expect}"


print("working directory:", WORK)

## 1. The Measurement Set

`examples/a1_simtruth.py generate small DIR` writes `DIR/a1-small.ms`. Next
to it go the truth (`truth.json`, `truth.npz`), the RFI mask
(`a1-small.rfi_mask.npz`) and the provenance with seeds and hashes.

In [ ]:
sh(f'python "{ROOT / "examples" / "a1_simtruth.py"}" generate small . --workers 2', tail=1)
MS = WORK / "a1-small.ms"
truth = json.loads((WORK / "truth.json").read_text(encoding="utf-8"))
sorted(p.name for p in WORK.iterdir())

What is in it, read with rang's own MS reader (`rang.core_ms`): one row per
baseline and integration, cells of 64 channels × 4 correlations.

In [ ]:
with core_ms.open_table(str(MS)) as t:
    columns = t.colnames()
    nrow = len(t.getcol("TIME"))
    field_id, scan = t.getcol("FIELD_ID"), t.getcol("SCAN_NUMBER")
    data_shape = t.getcol("DATA", 0, 1).shape[1:]
with core_ms.open_table(str(MS / "FIELD")) as t:
    field_names = list(t.getcol("NAME"))
with core_ms.open_table(str(MS / "ANTENNA")) as t:
    antennas = list(t.getcol("NAME"))
print(f"{nrow} rows, {len(antennas)} antennas, DATA cells {data_shape} (channel, correlation)")
print("columns:", ", ".join(columns))
pd.DataFrame(
    [
        {"field": name, "role": f["role"], "scans": sorted(set(scan[field_id == i].tolist())), "rows": int((field_id == i).sum())}
        for i, (name, f) in enumerate(zip(field_names, truth["fields"]))
    ]
)

## 2. The configuration

One YAML file, schema `rang-config` version 1, drives every stage. The file
used here is {gh}`examples/ms-to-image/config.yaml`; the CI job runs the
same file. `rang-core config defaults` prints every key with its default and
a comment, and the full list is on the {doc}`../../cli` page. Keys you leave
out take their defaults.

In [ ]:
for name in ("config.yaml", "strategy.yaml"):
    shutil.copy(ROOT / "examples" / "ms-to-image" / name, WORK / name)
print((WORK / "config.yaml").read_text(encoding="utf-8"))

Section by section:

`fields`
: FIELD names (or ids) by role: `fcal` (flux), `bpcal` (bandpass, defaulting
  to the `fcal` fields), `gcal` (gain), `xcal` (polarisation angle) and
  `target`. Every other section refers to roles, so the same file serves
  another observation once these lines change. A name that matches no FIELD
  row, or matches two, is refused.

`data` (not set here)
: `weights` (`as_is`: WEIGHT_SPECTRUM, else WEIGHT per channel; `uniform`: CARACal's
  `prep.specweights`), `scans` (a SCAN_NUMBER selection) and `memory_limit_gb`
  (8 GiB by default). Flag, calibrate and apply read the MS in chunks within
  that limit, so an MS larger than memory runs.

`flag`
: the Tricolour 0.2.1 port on DATA, every field. `strategy` is a Tricolour
  strategy file; the path is relative to the configuration. On real data use
  CARACal's `mk_rfi_flagging_calibrator_fields_firstpass.yaml` or your own.
  The tutorial strategy (printed below) is written for A1 small's coarse
  channels. `write: replace` (the default) makes FLAG the strategy's flags, as
  Tricolour does; `or` keeps the flags already there.

`calibrate`
: `model` writes the flux calibrator's model to MODEL_DATA, here CASA setjy's
  Stevens-Reynolds 2016 model of PKS 1934−638. The other sources are `unity`,
  `manual`, `lsm` or an existing column. `refant`, `uvrange` and the
  CASA-style `minsnr` and `minblperant` apply to every solve. `primary`
  solves K (delay), G (gain) and B (bandpass) on the flux calibrator, in
  order, each with its earlier terms pre-applied. `secondary` solves G on
  the gain calibrator and F, the flux scale (CASA `fluxscale`): Gsec
  rescaled to the flux calibrator's scale. Each term is CASA's, writes a
  CASA calibration table and takes CASA's `solint`, `combine`, `calmode`,
  `preapply`/`interp`/`gainfield`, `fillgaps` and `smodel`. A `polcal`
  section (Gpol, KCROSS, Xf, Df: CARACal's polarisation strategy 1) is left
  out here.

`apply`
: which tables are applied to which fields, written to CORRECTED_DATA. The
  flux calibrator gets its own K, G, B. The gain calibrator and the target
  get K and B from J1939−6342 and F from the nearest gain-calibrator scan,
  interpolated in time. `applymode: calflag` flags data without a valid
  solution, as CASA does. `parang` corrects the parallactic angle (needed
  with polcal terms).

`image`
: one GeMClean image per field, of CORRECTED_DATA once this run's apply has
  finished: 1024² pixels of 3″ (51′, which holds every target component),
  Briggs 0, at most 20,000 components, stopping at 3 robust σ of the
  residual, 2 threads. `write_model: true` predicts the CLEAN model into
  MODEL_DATA, as WSClean does. Spectral output (`channels_out`,
  `join_channels`, `fit_spectral_pol`), masks, tapers, multiscale and Stokes
  IQUV have their own keys.

`selfcal` (not run here)
: CARACal's rounds of image, model, solve (Sp, Sa or Sd) and apply. On A1
  the unmasked loop does not meet its truth limits
  ({gh}`research/c5-selfcal.md`, {gh}`research/c5b-selfcal-masks.md`).

`run`
: the stages `rang-core run` executes, always in the order flag, calibrate,
  apply, image, selfcal.

The flagging strategy:

In [ ]:
print((WORK / "strategy.yaml").read_text(encoding="utf-8"))

`rang-core config check CONFIG MS` validates the whole file against the
schema and the MS before anything is written. It reports every problem at
once and exits 2 if there is one.

In [ ]:
sh("rang-core config check config.yaml a1-small.ms")

## 3. The run

`rang-core run config.yaml a1-small.ms` runs the stages in order. The MS is
written in place: FLAG, MODEL_DATA and CORRECTED_DATA. Everything else goes
into a run directory beside it, `a1-small.ms.rang-run`, unless `--run-dir`
says otherwise.

In [ ]:
sh("rang-core run config.yaml a1-small.ms")
RUN = WORK / "a1-small.ms.rang-run"

The run directory: the state file, the resolved configuration, one folder
per stage with a `summary.json`, the calibration tables, the images and the
receipts.

In [ ]:
for path in sorted(RUN.rglob("*")):
    depth = len(path.relative_to(RUN).parts)
    if depth <= 3 and not (path.parent.name in "KGBF" or path.parent.name in ("Gsec", "F.append")):
        print("  " * (depth - 1) + path.name + ("/" if path.is_dir() else ""))

In [ ]:
state = json.loads((RUN / "state.json").read_text(encoding="utf-8"))
pd.DataFrame(
    [
        {"stage": s, "status": e["status"], "seconds": round(e["seconds"], 1), "flag version saved before": e.get("flag_version") or "– (writes no flags)", "config sha256": e["config_sha256"][:12]}
        for s, e in state["stages"].items()
    ]
)

### 3.1 Resume

Each stage records the hash of the configuration sections it depends on. Run
the same command again and every finished stage with an unchanged hash is
skipped. A failed or interrupted stage runs again. A stage whose
configuration changed is refused unless you pass `--force`; a forced `flag`
first restores the flags saved before its earlier attempt.

In [ ]:
sh("rang-core run config.yaml a1-small.ms")

## 4. The outputs

### 4.1 FLAG and its backups

Before the first write ever made to an MS, rang-core saves its flags as the
CASA flag version `rang_original`. Each writing stage then saves one more,
`rang_<stage>_<UTC time>`, before it writes. They live in
`a1-small.ms.flagversions/`, the same place and format CASA's `flagmanager`
uses. Restore one with `flagmanager(vis, mode='restore', versionname=...)` in
CASA, or with `rang.formats.flagversions.restore(ms, name)`.

In [ ]:
pd.DataFrame(flagversions.versions(MS), columns=["version", "comment"])

In [ ]:
with core_ms.open_table(str(MS)) as t:
    flag = t.getcol("FLAG")
cell_flag = flag.any(-1)
rfi = {k: np.unpackbits(v, axis=1, count=64).astype(bool) for k, v in np.load(WORK / "a1-small.rfi_mask.npz").items() if k in ("any", "above_5sigma")}
print(f"flagged after the run: {flag.mean():.3f} of all cells")
pd.DataFrame(
    [
        {"truth mask": k, "cells in mask": f"{m.mean():.3f}", "recall (mask cells flagged)": f"{cell_flag[m].mean():.3f}", "false flags (clean cells flagged)": f"{cell_flag[~m].mean():.3f}"}
        for k, m in rfi.items()
    ]
)

Measured against A1's RFI mask, almost all of the work here is done by the
static mask on baselines up to 1 km. On channels of 13.4 MHz it covers most
of the GSM, GNSS and satellite bands. The SumThreshold step alone recalls
3–5 % of the simulated RFI on this MS, so this table is not a flagging
result. The flagging studies are notebooks 2 and 5,
{gh}`research/flagging-verdict.md` and {gh}`research/f3-flagger.md`.

Restoring `rang_original` on a copy undoes every stage's flags:

In [ ]:
copy = WORK / "restored.ms"
shutil.copytree(MS, copy)
shutil.copytree(flagversions.root(MS), flagversions.root(copy))
flagversions.restore(copy, "rang_original")
with core_ms.open_table(str(copy)) as t:
    print("flagged after restoring rang_original:", t.getcol("FLAG").mean())
shutil.rmtree(copy)
shutil.rmtree(flagversions.root(copy))

### 4.2 The calibration tables

`calibrate/` holds one CASA table per term (K, G, B, Gsec, F, plus F's
`F.append`). CASA's `applycal`, `plotms` and `python-casacore` read them as
they are. Here are the K table's delays against A1's true delays, both
relative to the reference antenna m000:

In [ ]:
with core_ms.open_table(str(RUN / "calibrate" / "K")) as t:
    delay, antenna1 = t.getcol("FPARAM")[:, 0, :], t.getcol("ANTENNA1")
true_delay = np.load(WORK / "truth.npz")["gauge0_tau_ns"][antenna1]
err = np.abs(delay - true_delay)
print(f"{len(delay)} solutions (16 antennas × 2 flux-calibrator scans), true |τ| 1-10 ns")
print(f"|K − truth|: median {np.median(err):.3f} ns, max {err.max():.3f} ns")

### 4.3 MODEL_DATA and CORRECTED_DATA

MODEL_DATA holds two models. On the flux calibrator it holds the setjy model
the solves used. On the target, with `image.write_model`, it holds the CLEAN
model predicted after imaging; `selfcal` would solve against that. On the
other fields it is zero. CORRECTED_DATA is DATA with the tables of the
`apply` section applied. On the flux calibrator, corrected amplitudes should
follow its model spectrum (about 14-15 Jy across the band); the raw ones
carry the bandpass:

In [ ]:
fcal = field_names.index("J1939-6342")
with core_ms.open_table(str(MS)) as t:
    rows = np.flatnonzero(field_id == fcal)
    cols = {c: t.getcol(c, int(rows[0]), int(rows[-1] - rows[0] + 1)) for c in ("DATA", "CORRECTED_DATA", "MODEL_DATA", "FLAG")}
sel = field_id[rows[0] : rows[-1] + 1] == fcal
with core_ms.open_table(str(MS / "SPECTRAL_WINDOW")) as t:
    freq = t.getcol("CHAN_FREQ")[0] / 1e6


def stokes_i_amp(c):
    v = np.ma.masked_array(cols[c][sel][..., [0, 3]], cols["FLAG"][sel][..., [0, 3]])
    return np.abs(v).mean(axis=(0, 2))


fig, ax = plt.subplots(figsize=(8, 3.2))
for c, style in (("DATA", "C0-"), ("CORRECTED_DATA", "C1-"), ("MODEL_DATA", "k--")):
    ax.plot(freq, stokes_i_amp(c), style, label=c)
ax.set(xlabel="frequency (MHz)", ylabel="mean |XX|, |YY| (Jy)", title="J1939−6342, unflagged cells")
ax.legend()
plt.show()

### 4.4 The image

`image/SIM-TARGET/` holds GeMClean's FITS files with WSClean's names:
`image-image.fits` (restored), `-model`, `-residual`, `-dirty` and `-psf`.
Alongside them are `image-report.json` and `gemclean.log`. The image stage's
`summary.json` records the exact `gemclean` command line and the SHA-256 of
every file.

In [ ]:
folder = RUN / "image" / "SIM-TARGET"
with fits.open(folder / "image-image.fits") as hdul:
    header, restored = hdul[0].header, hdul[0].data.squeeze()
residual = fits.getdata(folder / "image-residual.fits").squeeze()
wcs = WCS(header).celestial
nu = header["CRVAL3"]
print(f"restoring beam {header['BMAJ'] * 3600:.1f}″ × {header['BMIN'] * 3600:.1f}″ at {header['BPA']:.0f}°, {nu / 1e6:.0f} MHz")
print(f"residual rms {np.std(residual) * 1e3:.2f} mJy/beam, peak {restored.max():.3f} Jy/beam")


def compare(image, k=6):
    rows = []
    for c in sorted(truth["target_components"], key=lambda c: -c["flux"])[:k]:
        x, y = (int(round(float(v))) for v in wcs.world_to_pixel_values(c["ra_deg"], c["dec_deg"]))
        rows.append({
            "kind": c["kind"], "FWHM (″)": round(c["fwhm_arcsec"]),
            "true S (Jy)": round(c["flux"] * (nu / c["ref_freq_hz"]) ** c["alpha"], 4),
            "image peak (Jy/beam)": round(float(image[y - 2 : y + 3, x - 2 : x + 3].max()), 4),
        })
    return pd.DataFrame(rows)


compare(restored)

The points come back at their true flux density. Fluxes are apparent: A1
simulates no primary beam. The Gaussians' peaks are much lower than their
integrated flux because a 25″ Gaussian spreads over several 29″ × 5″ beams.
The beam is elongated because a 16-antenna array observing at Dec −36° for
three hours has sparse uv coverage.

The noise per visibility is known: σ = SEFD / √(2 Δν Δt), 28.7 mJy per real
part and correlation. The residual can therefore be compared with the
thermal noise of a naturally weighted Stokes I image of the unflagged
target cells. Briggs 0 weighting raises the thermal level somewhat. A1
also puts a gain error on the target alone, the S term of its measurement
equation, which the calibrators cannot see. This notebook does not separate
the two causes:

In [ ]:
sigma = truth["design"]["sigma_jy"]
target = field_names.index("SIM-TARGET")
good = ~flag[field_id == target][..., [0, 3]].any(-1)  # (row, channel) cells with XX and YY
thermal = sigma / np.sqrt(2) / np.sqrt(good.sum())
s_term = np.load(WORK / "truth.npz")["raw_selfcal"][np.load(WORK / "truth.npz")["field"] == target]
print(f"natural-weighting thermal noise ≈ {thermal * 1e3:.3f} mJy/beam; residual rms {np.std(residual) * 1e3:.2f} mJy/beam ({np.std(residual) / thermal:.0f}×)")
print(f"A1's S term on the target: phase {np.degrees(np.angle(s_term)).std():.1f}° rms, amplitude {np.abs(s_term).std() * 100:.1f} % rms")

Removing S is the job of self-calibration (the `selfcal` section). On A1 it
does not yet meet its truth limits ({gh}`research/c5-selfcal.md`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), subplot_kw={"projection": wcs})
for ax, (title, im) in zip(axes, (("restored, CORRECTED_DATA", restored), ("residual", residual))):
    shown = ax.imshow(im, origin="lower", cmap="inferno", vmin=-3e-3, vmax=3e-2 if im is restored else 3e-3)
    ax.set_title(title)
    plt.colorbar(shown, ax=ax, fraction=0.046, label="Jy/beam")
plt.show()

### 4.5 `gemclean` on its own

`gemclean` is the imager the image stage runs. It installs with the wheel as
its own command, with WSClean-style options (`gemclean -h`). Run it by hand
for anything the `image` section does not cover. Here it is on the
uncalibrated DATA column of the same field (id 3) with the same settings,
which shows what calibration bought:

In [ ]:
target = field_names.index("SIM-TARGET")
sh(f"gemclean -name raw -size 1024 1024 -scale 3asec -weight briggs 0 -niter 20000 -auto-threshold 3 -j 2 -data-column DATA -field {target} -no-update-model-required a1-small.ms", tail=3)
raw = fits.getdata(WORK / "raw-image.fits").squeeze()
print(f"raw residual rms {np.std(fits.getdata(WORK / 'raw-residual.fits')) * 1e3:.2f} mJy/beam")
compare(raw)

### 4.6 The receipt

Every command writes a receipt to `<run>/receipts/` (schema `rang.receipt`,
version 4). It records the command and its arguments, the inputs with their
hashes, the environment (Python, the platform, rang's version and every
`RANG_*` variable) and the git commit, which is null outside a source
checkout. With the receipt, `state.json` and the configuration, a run can be
reproduced and audited.

In [ ]:
receipt = json.loads(sorted((RUN / "receipts").glob("run-*.json"))[0].read_text(encoding="utf-8"))
print("keys:", ", ".join(receipt))
print(json.dumps({k: receipt[k] for k in ("schema", "version", "command", "commit", "exit") if k in receipt}, indent=1)[:1200])

## 5. What rang-core refuses

rang-core checks the data and the configuration before the first write.
When it refuses, it exits 2, names what it found and the task or note that
covers it, and leaves the MS unchanged. Two examples follow.

**Unsupported data.** MeerKAT has linear feeds. Relabel a copy's
correlations as circular (RR, RL, LR, LL) and `flag` refuses it:

In [ ]:
from rang import core_ms_write

circular = WORK / "circular.ms"
shutil.copytree(MS, circular)
with core_ms_write.open_table(str(circular / "POLARIZATION"), readonly=False) as t:
    t.putcell("CORR_TYPE", 0, np.array([5, 6, 7, 8], np.int32))  # RR, RL, LR, LL
sh("rang-core flag config.yaml circular.ms", expect=2)
print("flag versions written:", flagversions.root(circular).exists())
shutil.rmtree(circular)

The other refused data cases (more than one spectral window in use, cells of
different shapes, two or one correlations, missing FLAG or FLAG_ROW, a field
id without a FIELD row) are listed with their messages in
{gh}`research/w4-unsupported.md`.

**A setting not implemented yet.** The schema can express things rang
cannot do yet. These are refused by name, not ignored. For example, writing
a model for a Stokes IQUV image:

In [ ]:
text = (WORK / "config.yaml").read_text(encoding="utf-8")
(WORK / "iquv.yaml").write_text(text.replace("  write_model: true", "  write_model: true\n  pol: IQUV"), encoding="utf-8")
sh("rang-core config check iquv.yaml a1-small.ms", expect=2)

## 6. On a real MeerKAT observation

The commands are the same. Change `fields` to your FIELD names, use
CARACal's flagging strategy and image settings that suit the field. Full
polarisation needs `calibrate.polcal` and `apply.parang`; set
`data.memory_limit_gb` to what your machine has.
{gh}`tests/data/w2/e2a.yaml` is E2-A's configuration for ESO 137-006 in
Stokes I, and {gh}`tests/data/w2/e2b.yaml` E2-B's in full polarisation.

```{important}
**Placeholder, to be filled from task E2** ({gh}`research/finish-plan.md`;
E2 is running and has no results yet). On the public ESO 137-006 observation
this section will quote:

- **[E2: runtime]** wall time per stage of `rang-core run` and the machine;
- **[E2: flags]** the flagged fraction against CARACal's;
- **[E2: calibration]** the calibration tables against CASA's and CARACal's;
- **[E2: image]** image rms, dynamic range and source fluxes against
  CARACal and WSClean.

Until then nothing here claims anything about real data. The A1 numbers
above are simulated.
```

**Where next:** notebooks 1–4 look inside each step on real MeerKAT data;
{doc}`../../cli` lists every command and option; {gh}`research/w2-commands.md`
describes the command line and its resume rules.